# UC04 — chuẩn bị dữ liệu chuỗi trên CPU

Chạy sau notebook pilot. Dữ liệu numeric và nhãn dùng cùng protocol; không mở final test. Cài dependencies theo `docs/SEQUENCE_RUNBOOK.md`.

In [ ]:
from pathlib import Path
import sys
import os

# Sửa SOURCE_ROOT nếu dùng source được gắn qua Kaggle Input.
SOURCE_ROOT = Path.cwd()
if not (SOURCE_ROOT / "src").is_dir() and (SOURCE_ROOT.parent / "src").is_dir():
    SOURCE_ROOT = SOURCE_ROOT.parent
# SOURCE_ROOT = Path("/kaggle/input/safeanes-source")
assert (SOURCE_ROOT / "src" / "safeanes").is_dir(), "Đặt SOURCE_ROOT tới repository"
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else SOURCE_ROOT
sys.path.insert(0, str(SOURCE_ROOT / "src"))
# Không đưa .local_deps Windows sang Kaggle.
if os.name == "nt" and (SOURCE_ROOT / ".local_deps").is_dir():
    sys.path.insert(0, str(SOURCE_ROOT / ".local_deps"))
print("Source:", SOURCE_ROOT, "Output:", WORK_ROOT)


In [ ]:
from safeanes.sequences import build_sequences, SequenceStore, load_dataset
DATASET = WORK_ROOT / "data/pilot_v1"
RAW_ROOT = WORK_ROOT / "data/vitaldb"
SEQUENCES = WORK_ROOT / "data/sequences_v1"
# Có thể đặt DATASET và RAW_ROOT vào Kaggle Input; output SEQUENCES phải writable.
if (SEQUENCES / "sequences.json").is_file():
    store = SequenceStore(SEQUENCES, DATASET)  # xác minh toàn bộ hashes
    print("Reusing verified cache:", len(store.meta["cases"]), "cases")
else:
    print(build_sequences(DATASET, RAW_ROOT, SEQUENCES))


In [ ]:
import pandas as pd
meta, protocol, manifest, windows = load_dataset(DATASET)
display(pd.read_csv(DATASET / "quality.csv"))
display(windows.groupby("eligible").size().rename("decisions"))
print("Cases:", len(manifest), "Subjects:", manifest.subjectid.nunique())
print("Dataset SHA-256:", meta["windows_sha256"])


Lưu `pilot_v1` và `sequences_v1` thành Input riêng cho notebook GPU. Không cần tải raw vào phiên GPU. Khi mở rộng cohort, tạo dataset/cache mới và không so hai split khác nhau như ablation.